# ColdLink AI - Model Training and Evaluation
## Training multiple models with temporal validation and comprehensive metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import json
from pathlib import Path
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
import xgboost as xgb
import shap

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Engineered Data

In [ ]:
# Load data
df = pd.read_csv('../data/engineered_features.csv')
df['date'] = pd.to_datetime(df['date'])

# Sort by date for temporal split
df = df.sort_values(['date', 'batch_id']).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nFailure rate: {df['target'].mean()*100:.2f}%")

## 2. Prepare Features and Target

In [ ]:
# Define features to exclude
exclude_cols = ['date', 'batch_id', 'target', 'location', 'current_hop', 'external_storage']

# Separate numeric and categorical features
categorical_features = ['location', 'current_hop', 'external_storage']
numeric_features = [col for col in df.columns if col not in exclude_cols and col not in categorical_features]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"\nNumeric features sample:")
print(numeric_features[:10])

In [ ]:
# Encode categorical features
label_encoders = {}
df_encoded = df.copy()

for col in categorical_features:
    le = LabelEncoder()
    df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {df_encoded[col].nunique()} unique values")

# Update feature list with encoded columns
encoded_categorical = [col + '_encoded' for col in categorical_features]
all_features = numeric_features + encoded_categorical

print(f"\nTotal features for modeling: {len(all_features)}")

## 3. Chronological Train/Validation/Test Split

**Critical**: Use time-based splitting to prevent temporal leakage
- Train: First 60% of time period
- Validation: Next 20% of time period  
- Test: Final 20% of time period

In [ ]:
# Calculate time-based split points
min_date = df_encoded['date'].min()
max_date = df_encoded['date'].max()
date_range = (max_date - min_date).total_seconds()

# Calculate split dates
train_end = min_date + pd.Timedelta(seconds=date_range * 0.6)
val_end = min_date + pd.Timedelta(seconds=date_range * 0.8)

# Split data
train_df = df_encoded[df_encoded['date'] <= train_end]
val_df = df_encoded[(df_encoded['date'] > train_end) & (df_encoded['date'] <= val_end)]
test_df = df_encoded[df_encoded['date'] > val_end]

print("Chronological Data Split:")
print("=" * 70)
print(f"\nTrain Set:")
print(f"  Date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"  Size: {len(train_df):,} records ({len(train_df)/len(df_encoded)*100:.1f}%)")
print(f"  Batches: {train_df['batch_id'].nunique()}")
print(f"  Failure rate: {train_df['target'].mean()*100:.2f}%")

print(f"\nValidation Set:")
print(f"  Date range: {val_df['date'].min()} to {val_df['date'].max()}")
print(f"  Size: {len(val_df):,} records ({len(val_df)/len(df_encoded)*100:.1f}%)")
print(f"  Batches: {val_df['batch_id'].nunique()}")
print(f"  Failure rate: {val_df['target'].mean()*100:.2f}%")

print(f"\nTest Set:")
print(f"  Date range: {test_df['date'].min()} to {test_df['date'].max()}")
print(f"  Size: {len(test_df):,} records ({len(test_df)/len(df_encoded)*100:.1f}%)")
print(f"  Batches: {test_df['batch_id'].nunique()}")
print(f"  Failure rate: {test_df['target'].mean()*100:.2f}%")

# Extract X and y for each set
X_train = train_df[all_features]
y_train = train_df['target']

X_val = val_df[all_features]
y_val = val_df['target']

X_test = test_df[all_features]
y_test = test_df['target']

print("\n✓ Temporal split complete - no future leakage")

## 4. Feature Scaling

In [ ]:
# Scale numeric features (fit only on training data)
scaler = StandardScaler()

# Identify which columns to scale (numeric features only)
numeric_feature_indices = [i for i, col in enumerate(all_features) if col in numeric_features]

# Fit scaler on training data
X_train_scaled = X_train.copy()
X_train_scaled.iloc[:, numeric_feature_indices] = scaler.fit_transform(X_train.iloc[:, numeric_feature_indices])

# Transform validation and test sets
X_val_scaled = X_val.copy()
X_val_scaled.iloc[:, numeric_feature_indices] = scaler.transform(X_val.iloc[:, numeric_feature_indices])

X_test_scaled = X_test.copy()
X_test_scaled.iloc[:, numeric_feature_indices] = scaler.transform(X_test.iloc[:, numeric_feature_indices])

print("✓ Features scaled (StandardScaler)")
print(f"✓ Scaler fitted on training data only")

## 5. Train Multiple Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),  # Handle imbalance
        n_jobs=-1,
        eval_metric='logloss'
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE
    )
}

print("Models to train:")
for name in models.keys():
    print(f"  • {name}")

In [ ]:
# Train all models
trained_models = {}
training_history = {}

print("Training models...\n")
print("=" * 70)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for Logistic Regression, raw for tree-based
    if name == 'Logistic Regression':
        X_tr, X_v = X_train_scaled, X_val_scaled
    else:
        X_tr, X_v = X_train, X_val
    
    # Train model
    model.fit(X_tr, y_train)
    
    # Make predictions
    y_train_pred = model.predict(X_tr)
    y_train_proba = model.predict_proba(X_tr)[:, 1]
    
    y_val_pred = model.predict(X_v)
    y_val_proba = model.predict_proba(X_v)[:, 1]
    
    # Calculate metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    
    train_f1 = f1_score(y_train, y_train_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    
    train_roc = roc_auc_score(y_train, y_train_proba)
    val_roc = roc_auc_score(y_val, y_val_proba)
    
    # Store results
    trained_models[name] = model
    training_history[name] = {
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'train_roc_auc': train_roc,
        'val_roc_auc': val_roc
    }
    
    print(f"  Train Accuracy: {train_acc:.4f} | Val Accuracy: {val_acc:.4f}")
    print(f"  Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")
    print(f"  Train ROC-AUC: {train_roc:.4f} | Val ROC-AUC: {val_roc:.4f}")
    print(f"  ✓ {name} training complete")

print("\n" + "=" * 70)
print("All models trained successfully!")

## 6. Comprehensive Model Evaluation

In [ ]:
def calculate_comprehensive_metrics(y_true, y_pred, y_proba):
    """Calculate all evaluation metrics"""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
        'pr_auc': average_precision_score(y_true, y_proba),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'fnr': fn / (fn + tp) if (fn + tp) > 0 else 0,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn
    }
    return metrics

# Evaluate all models on test set
test_results = {}

print("Test Set Evaluation:")
print("=" * 70)

for name, model in trained_models.items():
    print(f"\n{name}:")
    print("-" * 70)
    
    # Use appropriate data
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test
    
    # Predictions
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    
    # Calculate metrics
    metrics = calculate_comprehensive_metrics(y_test, y_pred, y_proba)
    test_results[name] = metrics
    
    # Print metrics
    print(f"  Accuracy:    {metrics['accuracy']:.4f}")
    print(f"  Precision:   {metrics['precision']:.4f}")
    print(f"  Recall:      {metrics['recall']:.4f}")
    print(f"  F1-Score:    {metrics['f1']:.4f}")
    print(f"  ROC-AUC:     {metrics['roc_auc']:.4f}")
    print(f"  PR-AUC:      {metrics['pr_auc']:.4f}")
    print(f"  Specificity: {metrics['specificity']:.4f}")
    print(f"  FPR:         {metrics['fpr']:.4f}")
    print(f"  FNR:         {metrics['fnr']:.4f}")
    print(f"\n  Confusion Matrix: TN={metrics['tn']}, FP={metrics['fp']}, FN={metrics['fn']}, TP={metrics['tp']}")

print("\n" + "=" * 70)

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame(test_results).T
comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'specificity', 'fpr', 'fnr']]

print("\nModel Comparison (Test Set):")
print("=" * 70)
print(comparison_df.round(4).to_string())

# Save comparison
comparison_df.to_csv('../reports/model_comparison.csv')
print("\n✓ Model comparison saved to reports/model_comparison.csv")

## 7. Visualize Model Performance

In [ ]:
# Plot ROC curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
for name, model in trained_models.items():
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test
    y_proba = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves - Test Set', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
for name, model in trained_models.items():
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test
    y_proba = model.predict_proba(X_te)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    axes[1].plot(recall, precision, label=f'{name} (AP = {pr_auc:.3f})', linewidth=2)

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', label='Baseline', linewidth=1)
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves - Test Set', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/roc_pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (name, model) in enumerate(trained_models.items()):
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Success', 'Failure'],
                yticklabels=['Success', 'Failure'])
    axes[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('../reports/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot metrics comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
comparison_subset = comparison_df[metrics_to_plot]

fig, ax = plt.subplots(figsize=(12, 6))
comparison_subset.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Model Performance Comparison - Test Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'], rotation=0)
ax.legend(title='Model', loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig('../reports/metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Select Best Model

In [ ]:
# Select best model based on F1-score (balance of precision and recall)
best_model_name = comparison_df['f1'].idxmax()
best_model = trained_models[best_model_name]
best_metrics = test_results[best_model_name]

print("=" * 70)
print("BEST MODEL SELECTION")
print("=" * 70)
print(f"\nBest Model: {best_model_name}")
print(f"Selection Criteria: Highest F1-Score")
print(f"\nPerformance on Test Set:")
print("-" * 70)
for metric, value in best_metrics.items():
    if metric not in ['tp', 'tn', 'fp', 'fn']:
        print(f"  {metric.upper():15}: {value:.4f}")

print("\n" + "=" * 70)

## 9. SHAP Explainability Analysis

In [ ]:
print("Calculating SHAP values...")
print("=" * 70)

# Use a sample for SHAP (computational efficiency)
X_test_sample = X_test.sample(min(1000, len(X_test)), random_state=RANDOM_STATE)
y_test_sample = y_test.loc[X_test_sample.index]

# Create SHAP explainer for best model
if 'Tree' in best_model_name or 'XG' in best_model_name or 'Hist' in best_model_name:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_sample)
    
    # For binary classification, get positive class SHAP values
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Positive class
else:
    # For Logistic Regression, use KernelExplainer (slower)
    X_train_summary = shap.sample(X_train_scaled, 100)
    explainer = shap.KernelExplainer(best_model.predict_proba, X_train_summary)
    shap_values = explainer.shap_values(X_test_sample[:100])  # Smaller sample for speed
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    X_test_sample = X_test_sample.head(100)

print(f"✓ SHAP values calculated for {len(X_test_sample)} samples")
print(f"✓ SHAP values shape: {shap_values.shape}")

In [ ]:
# Global feature importance
shap_importance = np.abs(shap_values).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'feature': all_features,
    'importance': shap_importance
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (SHAP):")
print("=" * 70)
print(feature_importance_df.head(20).to_string(index=False))

# Save feature importance
feature_importance_df.to_csv('../reports/shap_feature_importance.csv', index=False)
print("\n✓ SHAP feature importance saved")

In [ ]:
# Plot SHAP summary
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=20, show=False)
plt.title('SHAP Feature Importance - Test Set', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/shap_summary_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ SHAP summary plot saved")

In [ ]:
# Plot SHAP bar plot (feature importance)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Feature Importance (Bar) - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/shap_bar_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ SHAP bar plot saved")

## 10. Save Models and Artifacts

In [ ]:
# Create models directory if it doesn't exist
Path('../models').mkdir(exist_ok=True)

# Save best model
joblib.dump(best_model, '../models/best_model.pkl')
print(f"✓ Best model saved: models/best_model.pkl")

# Save all trained models
for name, model in trained_models.items():
    model_filename = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, f'../models/{model_filename}')
    print(f"✓ {name} saved: models/{model_filename}")

# Save scaler
joblib.dump(scaler, '../models/scaler.pkl')
print(f"✓ Scaler saved: models/scaler.pkl")

# Save label encoders
joblib.dump(label_encoders, '../models/label_encoders.pkl')
print(f"✓ Label encoders saved: models/label_encoders.pkl")

# Save feature names
joblib.dump({
    'all_features': all_features,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'encoded_categorical': encoded_categorical
}, '../models/feature_names.pkl')
print(f"✓ Feature names saved: models/feature_names.pkl")

# Save SHAP explainer and values
joblib.dump({
    'explainer': explainer,
    'shap_values_sample': shap_values,
    'X_test_sample': X_test_sample,
    'feature_importance': feature_importance_df
}, '../models/shap_explainer.pkl')
print(f"✓ SHAP explainer saved: models/shap_explainer.pkl")

In [ ]:
# Save model metadata
metadata = {
    'best_model': best_model_name,
    'best_model_metrics': {k: float(v) for k, v in best_metrics.items()},
    'all_models_comparison': comparison_df.to_dict(),
    'training_date': pd.Timestamp.now().isoformat(),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'num_features': len(all_features),
    'target_distribution': {
        'train': {'0': int((y_train==0).sum()), '1': int((y_train==1).sum())},
        'val': {'0': int((y_val==0).sum()), '1': int((y_val==1).sum())},
        'test': {'0': int((y_test==0).sum()), '1': int((y_test==1).sum())}
    },
    'feature_list': all_features,
    'top_features': feature_importance_df.head(20)['feature'].tolist()
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
    
print(f"✓ Model metadata saved: models/model_metadata.json")

## 11. Final Summary

In [ ]:
print("\n" + "=" * 70)
print("MODEL TRAINING AND EVALUATION COMPLETE")
print("=" * 70)

print("\n📊 SUMMARY:")
print("-" * 70)
print(f"✓ Models trained: {len(trained_models)}")
print(f"✓ Best model: {best_model_name}")
print(f"✓ Best F1-Score: {best_metrics['f1']:.4f}")
print(f"✓ Best ROC-AUC: {best_metrics['roc_auc']:.4f}")
print(f"✓ Features used: {len(all_features)}")
print(f"✓ SHAP explainability implemented")

print("\n💾 SAVED ARTIFACTS:")
print("-" * 70)
print("  • Best model (best_model.pkl)")
print("  • All trained models (*.pkl)")
print("  • Scaler (scaler.pkl)")
print("  • Label encoders (label_encoders.pkl)")
print("  • Feature names (feature_names.pkl)")
print("  • SHAP explainer (shap_explainer.pkl)")
print("  • Model metadata (model_metadata.json)")
print("  • Comparison CSV (model_comparison.csv)")
print("  • SHAP importance CSV (shap_feature_importance.csv)")
print("  • Visualization PNGs (reports/)")

print("\n🎯 NEXT STEPS:")
print("-" * 70)
print("  1. Build FastAPI backend")
print("  2. Implement recommendation engine")
print("  3. Create React frontend")
print("  4. Integrate and test end-to-end")

print("\n" + "=" * 70)
print("🎉 READY FOR DEPLOYMENT!")
print("=" * 70)